<a href="https://colab.research.google.com/github/vikranthrach/IIT-Patna--AI-and-ML-Course/blob/main/Building_First_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import time
import os
from openai import OpenAI

In [ ]:
from google.colab import userdata
openaiapikey = userdata.get('OPENAI_API_KEY')
print(openaiapikey[:10])

sk-proj-T3


In [ ]:
client = OpenAI(api_key=openaiapikey)
print("Client Created Successfully!!")

Client Created Successfully!!


In [ ]:
response = client.chat.completions.create(
    model = "gpt-5.6",
    messages = [
        {"role":"user", "content": "What is the current weather of Hyderabad?"}
    ]
)
print(response.choices[0])
print(response.choices[0].message.content)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Do you mean Hyderabad, Telangana, India? I don’t have access to live weather data in this chat. For current conditions, check Google Weather, AccuWeather, or the India Meteorological Department at **mausam.imd.gov.in**.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))
Do you mean Hyderabad, Telangana, India? I don’t have access to live weather data in this chat. For current conditions, check Google Weather, AccuWeather, or the India Meteorological Department at **mausam.imd.gov.in**.


In [ ]:
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role":"user", "content": "What is the capital of Inida?"}
    ]
)
print(response.choices[0])
print(response.choices[0].message.content)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of India is New Delhi.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))
The capital of India is New Delhi.


In [ ]:
def get_weather(city: str):
  ''' Get weather for a city '''
  fake_weather = {
      "hyderabad": "25 Degrees, partly cloudy, Humidity: 65%",
      "kakinada": "35 Degrees, partly sunny, Humidity: 80%",
      "delhi": "32 Degrees, partly cloudy, Humidity: 45%",
      "pune": "19 Degrees, partly rainy, Humidity: 63%",
      "amsterdam": "23 Degrees, partly cloudy, Humidity: 40%",
      "rajahmundry": "37 Degrees, Sunny, Humidity: 89%",

  }

  city_lower = city.lower().strip()
  if city_lower in fake_weather:
    return fake_weather[city_lower]
  return f"Weather data not avaiablaes for the {city}"

print("Weather ", get_weather("Rotterdam"))


Weather  Weather data not avaiablaes for the Rotterdam


In [ ]:
def calculate(expression: str) -> str:
    """Evaluate a math expression and return the result."""
    try:
        # Only allow safe math operations
        allowed = set('0123456789+-*/.() ')
        if not all(c in allowed for c in expression):
            return "Error: Only numbers and +-*/() allowed"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Json Schemas for registering tools

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city, e.g. Hyderabad"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Use this for mathematical computations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to evaluate, e.g. '(25 * 4) + 100'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print("Tools are defined in a json format")

for tool in tools:
  name = tool["function"]["name"]
  desc = tool["function"]["description"]
  params = list(tool['function']['parameters']['properties'].keys())

  print(f"{name} ({" ".join(params)}) - {desc}")


Tools are defined in a json format
get_weather (city) - Get the current weather for a city.
calculate (expression) - Evaluate a mathematical expression. Use this for mathematical computations.


# Configure the LLM to use the Tools

In [ ]:
messages = [
    {"role":"system", "content": "You are a helpful assistant and use tools when needed!"},
    {"role": "user", "content": "What is the weather of Hyderabad?"}
]

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = messages,
    tools = tools
)

# Let's inspect what the LLM decided
choice = response.choices[0]
print(f"Finish reason: {choice.finish_reason}")
print(f"Content: {choice.message.content}")
print(f"Tool calls: {choice.message.tool_calls}")

if choice.message.tool_calls:
    tc = choice.message.tool_calls[0]
    print(f"\n── LLM Decision ──")
    print(f"  Tool:      {tc.function.name}")
    print(f"  Arguments: {tc.function.arguments}")
    print(f"  ID:        {tc.id}")

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_A5kkEmmmmUpDOWctb8z8oWKv', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')]

── LLM Decision ──
  Tool:      get_weather
  Arguments: {"city":"Hyderabad"}
  ID:        call_A5kkEmmmmUpDOWctb8z8oWKv


In [ ]:
choice = response.choices[0]
print(choice)

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_A5kkEmmmmUpDOWctb8z8oWKv', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')]))


In [ ]:
messages = [
    {"role":"system", "content": "You are a helpful assistant and use tools when needed!"},
    {"role": "user", "content": "What is the weather of Hyderabad?"}
]

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = messages,
    tools = tools
)

# Let's inspect what the LLM decided
choice = response.choices[0]
print(f"Finish reason: {choice.finish_reason}")
print(f"Content: {choice.message.content}")
print(f"Tool calls: {choice.message.tool_calls}")

if choice.message.tool_calls:
    tc = choice.message.tool_calls[0]
    print(f"\n── LLM Decision ──")
    print(f"  Tool:      {tc.function.name}")
    print(f"  Arguments: {tc.function.arguments}")
    print(f"  ID:        {tc.id}")


available_tools = {
    "calculate": calculate,
    "get_weather": get_weather
}

tool_call = choice.message.tool_calls[0]
print(tool_call)

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)
print(function_name, arguments)

result = available_tools[function_name](**arguments)
print(result)


messages.append(choice.message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result
})

final_respose = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = messages,
    tools = tools
)
print(final_respose.choices[0].message.content)

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_6wm5CMXkU4f0p6Q2rOEtiM76', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')]

── LLM Decision ──
  Tool:      get_weather
  Arguments: {"city":"Hyderabad"}
  ID:        call_6wm5CMXkU4f0p6Q2rOEtiM76
ChatCompletionMessageFunctionToolCall(id='call_6wm5CMXkU4f0p6Q2rOEtiM76', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')
get_weather {'city': 'Hyderabad'}
25 Degrees, partly cloudy, Humidity: 65%
The weather in Hyderabad is currently 25 degrees Celsius, partly cloudy, with a humidity level of 65%.


In [ ]:
# ──────────────────────────────────────
# THE AGENT LOOP — The core of every agent
# ──────────────────────────────────────

def run_agent(user_message, max_steps=5, verbose=True):
    """
    Run a simple agent that can use tools to answer questions.

    Args:
        user_message: The user's question
        max_steps: Hard stop — max tool calls before giving up
        verbose: Print each step for debugging
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the available tools to answer questions accurately. If you can answer without tools, do so directly."},
        {"role": "user", "content": user_message}
    ]

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"🧑 User: {user_message}")
        print(f"{'═' * 60}")

    for step in range(max_steps):
        # Ask the LLM
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )

        choice = response.choices[0]

        # Case 1: LLM is done — it has a final answer
        if choice.finish_reason == "stop":
            if verbose:
                print(f"\n✅ Step {step + 1}: Final answer ready")
            return choice.message.content

        # Case 2: LLM wants to use a tool
        if choice.message.tool_calls:
            # Add assistant message to history
            messages.append(choice.message)

            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n🔧 Step {step + 1}: Calling {func_name}({args})")

                # Run the tool
                if func_name in available_tools:
                    result = available_tools[func_name](**args)
                else:
                    result = f"Error: Unknown tool '{func_name}'"

                if verbose:
                    print(f"   📤 Result: {result}")

                # Send result back
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

    # Hard stop reached
    return "⚠️ Agent reached maximum steps without finishing."

print("✅ Agent function defined!")

✅ Agent function defined!


In [ ]:
# Test 2: Weather question (needs weather tool)
answer = run_agent("What's the weather like in Pune?")
print(f"\n🤖 Answer: {answer}")


════════════════════════════════════════════════════════════
🧑 User: What's the weather like in Pune?
════════════════════════════════════════════════════════════

🔧 Step 1: Calling get_weather({'city': 'Pune'})
   📤 Result: 19 Degrees, partly rainy, Humidity: 63%

✅ Step 2: Final answer ready

🤖 Answer: The weather in Pune is 19 degrees Celsius, partly rainy, with a humidity level of 63%.


In [ ]:
# Test 2: Weather question (needs weather tool)
answer = run_agent("What's the (45*67)+ (23*89)")
print(f"\n🤖 Answer: {answer}")


════════════════════════════════════════════════════════════
🧑 User: What's the (45*67)+ (23*89)
════════════════════════════════════════════════════════════

🔧 Step 1: Calling calculate({'expression': '(45*67) + (23*89)'})
   📤 Result: 5062

✅ Step 2: Final answer ready

🤖 Answer: The result of the calculation \((45 \times 67) + (23 \times 89)\) is 5062.


In [ ]:
# Test 2: Weather question (needs weather tool)
answer = run_agent("What's the weather like in Hyderabad?")
print(f"\n🤖 Answer: {answer}")

In [ ]:
# Test 2: Weather question (needs weather tool)
answer = run_agent("What's the weather in Rotterdam")
print(f"\n🤖 Answer: {answer}")


════════════════════════════════════════════════════════════
🧑 User: What's the weather in Rotterdam
════════════════════════════════════════════════════════════

🔧 Step 1: Calling get_weather({'city': 'Rotterdam'})
   📤 Result: Weather data not avaiablaes for the Rotterdam

✅ Step 2: Final answer ready

🤖 Answer: I'm sorry, but I can't retrieve the weather data for Rotterdam at the moment. You might want to check a reliable weather website or app for the latest updates.


# Build the CSV Agents

In [ ]:
# ──────────────────────────────────────
# Create a sample dataset (no file needed — inline)
# ──────────────────────────────────────
import csv
from io import StringIO

# Sample employee data
CSV_DATA = """name,department,salary,experience_years,city
Rahul,Engineering,75000,5,Hyderabad
Priya,Marketing,55000,3,Bangalore
Arjun,Engineering,82000,7,Hyderabad
Sneha,HR,48000,2,Mumbai
Vikram,Engineering,90000,9,Delhi
Anita,Marketing,60000,4,Bangalore
Karthik,HR,52000,3,Chennai
Deepa,Engineering,70000,4,Hyderabad
Suresh,Marketing,65000,6,Mumbai
Kavya,HR,50000,2,Bangalore
Ravi,Engineering,95000,10,Delhi
Meera,Marketing,58000,3,Chennai
Arun,Engineering,78000,6,Hyderabad
Lavanya,HR,55000,4,Bangalore
Sanjay,Marketing,62000,5,Mumbai
"""

# Parse into list of dicts
reader = csv.DictReader(StringIO(CSV_DATA.strip()))
data = list(reader)

print(f"Dataset: {len(data)} employees")
print(f"Columns: {list(data[0].keys())}")
print(f"\nFirst 3 rows:")
for row in data[:3]:
    print(f"  {row}")

Dataset: 15 employees
Columns: ['name', 'department', 'salary', 'experience_years', 'city']

First 3 rows:
  {'name': 'Rahul', 'department': 'Engineering', 'salary': '75000', 'experience_years': '5', 'city': 'Hyderabad'}
  {'name': 'Priya', 'department': 'Marketing', 'salary': '55000', 'experience_years': '3', 'city': 'Bangalore'}
  {'name': 'Arjun', 'department': 'Engineering', 'salary': '82000', 'experience_years': '7', 'city': 'Hyderabad'}


In [ ]:
# ──────────────────────────────────────
# Define CSV analysis tools
# ──────────────────────────────────────

def list_columns() -> str:
    """List all columns in the dataset with sample values."""
    result = []
    for col in data[0].keys():
        samples = [row[col] for row in data[:3]]
        result.append(f"{col}: {samples}")
    return "\n".join(result)

def get_stats(column: str) -> str:
    """Get statistics (mean, min, max, count) for a numeric column."""
    try:
        values = [float(row[column]) for row in data]
        return json.dumps({
            "column": column,
            "count": len(values),
            "mean": round(sum(values) / len(values), 2),
            "min": min(values),
            "max": max(values),
            "total": sum(values)
        })
    except (ValueError, KeyError) as e:
        return f"Error: {e}. Available numeric columns: salary, experience_years"

def filter_count(column: str, value: str) -> str:
    """Count rows where column equals value."""
    try:
        matches = [row for row in data if row[column].lower() == value.lower()]
        if matches:
            return json.dumps({
                "filter": f"{column} = {value}",
                "count": len(matches),
                "matching_names": [m['name'] for m in matches]
            })
        return f"No rows found where {column} = {value}"
    except KeyError:
        return f"Error: Column '{column}' not found. Available: {list(data[0].keys())}"

# Test
print(list_columns())
print()
print(get_stats("salary"))
print()
print(filter_count("department", "Engineering"))

name: ['Rahul', 'Priya', 'Arjun']
department: ['Engineering', 'Marketing', 'Engineering']
salary: ['75000', '55000', '82000']
experience_years: ['5', '3', '7']
city: ['Hyderabad', 'Bangalore', 'Hyderabad']

{"column": "salary", "count": 15, "mean": 66333.33, "min": 48000.0, "max": 95000.0, "total": 995000.0}

{"filter": "department = Engineering", "count": 6, "matching_names": ["Rahul", "Arjun", "Vikram", "Deepa", "Ravi", "Arun"]}


In [ ]:
# ──────────────────────────────────────
# JSON schemas for CSV tools
# ──────────────────────────────────────

csv_tools = [
    {
        "type": "function",
        "function": {
            "name": "list_columns",
            "description": "List all columns in the employee dataset with sample values. Use this first to understand what data is available.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_stats",
            "description": "Get statistics (mean, min, max, count, total) for a numeric column like 'salary' or 'experience_years'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {
                        "type": "string",
                        "description": "Name of the numeric column to analyze"
                    }
                },
                "required": ["column"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "filter_count",
            "description": "Count how many rows match a condition (column equals a value). Returns count and matching employee names.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {
                        "type": "string",
                        "description": "Column to filter on (e.g. 'department', 'city')"
                    },
                    "value": {
                        "type": "string",
                        "description": "Value to match (e.g. 'Engineering', 'Hyderabad')"
                    }
                },
                "required": ["column", "value"]
            }
        }
    }
]

csv_available_tools = {
    "list_columns": list_columns,
    "get_stats": get_stats,
    "filter_count": filter_count,
}

print("CSV Analyst tools ready:")
for t in csv_tools:
    print(f"  🔧 {t['function']['name']} — {t['function']['description'][:60]}...")

CSV Analyst tools ready:
  🔧 list_columns — List all columns in the employee dataset with sample values....
  🔧 get_stats — Get statistics (mean, min, max, count, total) for a numeric ...
  🔧 filter_count — Count how many rows match a condition (column equals a value...


In [ ]:
# ──────────────────────────────────────
# The CSV Analyst Agent
# ──────────────────────────────────────

def csv_analyst(question, verbose=True):
    """An agent that answers questions about the employee dataset."""
    messages = [
        {"role": "system", "content": (
            "You are a data analyst assistant. You have access to an employee dataset. "
            "Use the available tools to answer questions about the data. "
            "Always start by listing columns if you're unsure what's available. "
            "Give clear, concise answers with the numbers."
        )},
        {"role": "user", "content": question}
    ]

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"📊 Question: {question}")
        print(f"{'─' * 60}")

    max_steps = 10
    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=csv_tools,
        )

        choice = response.choices[0]

        if choice.finish_reason == "stop":
            if verbose:
                print(f"{'─' * 60}")
            return choice.message.content

        if choice.message.tool_calls:
            messages.append(choice.message)
            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    args_str = ', '.join(f"{k}={v}" for k, v in args.items())
                    print(f"  🔧 {func_name}({args_str})")

                if func_name in csv_available_tools:
                    result = csv_available_tools[func_name](**args)
                else:
                    result = f"Unknown tool: {func_name}"

                if verbose:
                    print(f"     → {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

    return "Could not answer within step limit."

print("✅ CSV Analyst Agent ready!")

✅ CSV Analyst Agent ready!


In [ ]:
# Let's ask questions!

answer = csv_analyst("What is the average salary in the company?")
print(f"\n🤖 {answer}")


════════════════════════════════════════════════════════════
📊 Question: What is the average salary in the company?
────────────────────────────────────────────────────────────
  🔧 list_columns()
     → name: ['Rahul', 'Priya', 'Arjun']
department: ['Engineering', 'Marketing', 'Engineering']
salary: ['75000', '55000', '82000']
experience_years: ['5', '3', '7']
city: ['Hyderabad', 'Bangalore', 'Hyderabad']
  🔧 get_stats(column=salary)
     → {"column": "salary", "count": 15, "mean": 66333.33, "min": 48000.0, "max": 95000.0, "total": 995000.0}
────────────────────────────────────────────────────────────

🤖 The average salary in the company is approximately $66,333.33.


In [ ]:
answer = csv_analyst("How many people work in Engineering? Who are they?")
print(f"\n🤖 {answer}")


════════════════════════════════════════════════════════════
📊 Question: How many people work in Engineering? Who are they?
────────────────────────────────────────────────────────────
  🔧 filter_count(column=department, value=Engineering)
     → {"filter": "department = Engineering", "count": 6, "matching_names": ["Rahul", "Arjun", "Vikram", "Deepa", "Ravi", "Arun"]}
────────────────────────────────────────────────────────────

🤖 There are 6 people who work in Engineering. Their names are Rahul, Arjun, Vikram, Deepa, Ravi, and Arun.


In [ ]:
answer = csv_analyst("Which department has higher salaries — Engineering or Marketing? By how much?")
print(f"\n🤖 {answer}")


════════════════════════════════════════════════════════════
📊 Question: Which department has higher salaries — Engineering or Marketing? By how much?
────────────────────────────────────────────────────────────
  🔧 list_columns()
     → name: ['Rahul', 'Priya', 'Arjun']
department: ['Engineering', 'Marketing', 'Engineering']
salary: ['75000', '55000', '82000']
experience_years: ['5', '3', '7']
city: ['Hyderabad', 'Bangalore', 'Hyderabad']
  🔧 get_stats(column=salary)
     → {"column": "salary", "count": 15, "mean": 66333.33, "min": 48000.0, "max": 95000.0, "total": 995000.0}
  🔧 filter_count(column=department, value=Engineering)
     → {"filter": "department = Engineering", "count": 6, "matching_names": ["Rahul", "Arjun", "Vikram", "Deepa", "Ravi", "Arun"]}
  🔧 filter_count(column=department, value=Marketing)
     → {"filter": "department = Marketing", "count": 5, "matching_names": ["Priya", "Anita", "Suresh", "Meera", "Sanjay"]}
  🔧 filter_count(column=department, value=Engineering)

In [ ]:
answer = csv_analyst("How many employees are based in Hyderabad?")
print(f"\n🤖 {answer}")


════════════════════════════════════════════════════════════
📊 Question: How many employees are based in Hyderabad?
────────────────────────────────────────────────────────────
  🔧 list_columns()
     → name: ['Rahul', 'Priya', 'Arjun']
department: ['Engineering', 'Marketing', 'Engineering']
salary: ['75000', '55000', '82000']
experience_years: ['5', '3', '7']
city: ['Hyderabad', 'Bangalore', 'Hyderabad']
  🔧 filter_count(column=city, value=Hyderabad)
     → {"filter": "city = Hyderabad", "count": 4, "matching_names": ["Rahul", "Arjun", "Deepa", "Arun"]}
────────────────────────────────────────────────────────────

🤖 There are 4 employees based in Hyderabad. The matching names are Rahul, Arjun, Deepa, and Arun.
